In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv
import os

# Load .env
load_dotenv(find_dotenv())

# Database URL
DB_URL = os.getenv("LOCAL_DATABASE_URL")

# Support PostgreSQL + psycopg2
if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace(
        "postgresql://",
        "postgresql+psycopg2://",
        1
    )

# Create database engine
engine = create_engine(DB_URL)

print("Database connection berhasil")
print(f"Target: {DB_URL.split('@')[1] if DB_URL else 'NONE'}")

Database connection berhasil
Target: localhost:5432/retail_analytics


In [3]:
silver_tables = pd.read_sql(
    """
    SELECT
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'silver'
    ORDER BY table_name
    """,
    engine
)

display(silver_tables)

,table_name
0,campaign_spend
1,cities
2,city_reference
3,customer_addresses
4,customer_profiles
5,customers
6,inventory_snapshots
7,order_items
8,order_promotions
9,orders


In [4]:
customer_daily_sources = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver'
      AND table_name IN (
          'orders',
          'order_items',
          'refund_events',
          'return_events',
          'support_events'
      )
    ORDER BY table_name, ordinal_position
    """,
    engine
)

display(customer_daily_sources)

,table_name,column_name,data_type
0,order_items,order_id,text
1,order_items,quantity,bigint
2,order_items,product_id,text
3,order_items,unit_price,double precision
4,order_items,order_item_id,text
5,order_items,source_row_id,text
6,order_items,item_discount_amount,double precision
7,order_items,ingested_at_utc,timestamp with time zone
8,orders,status,text
9,orders,order_id,text


In [5]:
customer_order_base = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ),

    order_items_summary AS (
        SELECT
            order_id,
            SUM(quantity) AS unit_quantity,
            SUM(quantity * unit_price) AS gross_revenue,
            SUM(item_discount_amount) AS discount_amount
        FROM silver.order_items
        GROUP BY order_id
    )

    SELECT
        o.order_id,
        o.customer_id,
        DATE(o.ordered_at_utc::timestamptz) AS metric_date,
        o.sales_channel,
        COALESCE(i.unit_quantity, 0) AS unit_quantity,
        COALESCE(i.gross_revenue, 0) AS gross_revenue,
        COALESCE(i.discount_amount, 0) AS discount_amount,
        COALESCE(o.shipping_revenue, 0) AS shipping_revenue

    FROM orders_dedup o
    LEFT JOIN order_items_summary i
        ON o.order_id = i.order_id
    """,
    engine
)

print("Rows:", len(customer_order_base))
print("Unique orders:", customer_order_base["order_id"].nunique())
print("Unique customers:", customer_order_base["customer_id"].nunique())

display(customer_order_base.head(10))

Rows: 10000
Unique orders: 10000
Unique customers: 2455


,order_id,customer_id,metric_date,sales_channel,unit_quantity,gross_revenue,discount_amount,shipping_revenue
0,ORD-000299,CUST-00552,2026-07-23,MOBILE_APP,5.0,681.90,5.0,7.5
1,ORD-005343,CUST-00095,2026-08-12,WEB,4.0,268.04,5.0,7.5
2,ORD-005158,CUST-00734,2026-06-30,WEB,6.0,1372.00,2.5,7.5
3,ORD-002601,CUST-00167,2026-08-20,STORE,8.0,1557.97,7.5,3.5
4,ORD-006850,CUST-00518,2026-09-11,WEB,5.0,419.11,0.0,0.0
5,ORD-005256,CUST-01362,2026-08-21,STORE,1.0,298.58,0.0,0.0
6,ORD-003129,CUST-01529,2026-07-06,WEB,2.0,783.54,0.0,0.0
7,ORD-008070,CUST-00247,2026-06-30,WEB,6.0,1926.75,0.0,12.0
8,ORD-009333,CUST-01271,2026-07-08,MOBILE_APP,3.0,341.82,0.0,0.0
9,ORD-004295,CUST-02308,2026-07-17,MARKETPLACE,5.0,421.27,0.0,7.5


In [6]:
refund_by_order = pd.read_sql(
    """
    WITH refund_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY event_id
                    ORDER BY ingested_at_utc DESC
                ) AS rn
            FROM silver.refund_events
            WHERE event_type = 'REFUND_COMPLETED'
        ) x
        WHERE rn = 1
    )

    SELECT
        "payload.order_id" AS order_id,
        SUM("payload.amount") AS refund_amount
    FROM refund_dedup
    GROUP BY "payload.order_id"
    """,
    engine
)

print("Refund orders:", refund_by_order["order_id"].nunique())
print("Total refund:", round(refund_by_order["refund_amount"].sum(), 2))

display(refund_by_order.head(10))

Refund orders: 1991
Total refund: 1549240.21


,order_id,refund_amount
0,ORD-000018,1063.59
1,ORD-000035,432.69
2,ORD-000038,275.21
3,ORD-000039,141.39
4,ORD-000044,340.56
5,ORD-000052,1260.65
6,ORD-000053,288.59
7,ORD-000058,3203.06
8,ORD-000066,299.07
9,ORD-000069,1072.71


In [7]:
return_by_order = pd.read_sql(
    """
    WITH return_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY event_id
                    ORDER BY ingested_at_utc DESC
                ) AS rn
            FROM silver.return_events
        ) x
        WHERE rn = 1
    )

    SELECT
        "payload.order_id" AS order_id,
        COUNT(DISTINCT "payload.return_id") AS return_count
    FROM return_dedup
    GROUP BY "payload.order_id"
    """,
    engine
)

print("Return orders:", return_by_order["order_id"].nunique())
print("Total return:", return_by_order["return_count"].sum())

display(return_by_order.head(10))

Return orders: 1372
Total return: 1372


,order_id,return_count
0,ORD-000003,1
1,ORD-000011,1
2,ORD-000016,1
3,ORD-000029,1
4,ORD-000034,1
5,ORD-000039,1
6,ORD-000043,1
7,ORD-000044,1
8,ORD-000049,1
9,ORD-000056,1


In [8]:
support_by_customer_date = pd.read_sql(
    """
    WITH support_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY event_id
                    ORDER BY ingested_at_utc DESC
                ) AS rn
            FROM silver.support_events
        ) x
        WHERE rn = 1
    )

    SELECT
        "payload.customer_id" AS customer_id,
        DATE(occurred_at_utc::timestamptz) AS metric_date,
        COUNT(DISTINCT "payload.ticket_id") AS support_contacts

    FROM support_dedup

    GROUP BY
        "payload.customer_id",
        DATE(occurred_at_utc::timestamptz)

    ORDER BY
        customer_id,
        metric_date
    """,
    engine
)

print(
    "Customer-date combinations:",
    len(support_by_customer_date)
)

print(
    "Unique customers with support:",
    support_by_customer_date["customer_id"].nunique()
)

print(
    "Total support contacts:",
    support_by_customer_date["support_contacts"].sum()
)

display(support_by_customer_date.head(10))

Customer-date combinations: 2843
Unique customers with support: 1187
Total support contacts: 2870


,customer_id,metric_date,support_contacts
0,CUST-00002,2026-08-29,1
1,CUST-00002,2026-09-14,1
2,CUST-00003,2026-07-06,1
3,CUST-00003,2026-07-07,1
4,CUST-00004,2026-08-06,1
5,CUST-00004,2026-08-18,1
6,CUST-00004,2026-09-19,1
7,CUST-00005,2026-08-29,1
8,CUST-00005,2026-09-06,1
9,CUST-00006,2026-08-19,1


In [9]:
customer_order_rank = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    )

    SELECT
        order_id,
        customer_id,
        ordered_at_utc,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY ordered_at_utc, order_id
        ) AS customer_order_rank

    FROM orders_dedup
    """,
    engine
)

print("Rows:", len(customer_order_rank))
print(
    "Unique orders:",
    customer_order_rank["order_id"].nunique()
)

display(customer_order_rank.head(10))

Rows: 10000
Unique orders: 10000


,order_id,customer_id,ordered_at_utc,customer_order_rank
0,ORD-006860,CUST-00001,2026-06-26T15:18:00Z,1
1,ORD-009732,CUST-00001,2026-07-18T01:09:00Z,2
2,ORD-002644,CUST-00001,2026-08-12T23:07:00Z,3
3,ORD-006550,CUST-00001,2026-08-19T06:24:00Z,4
4,ORD-009265,CUST-00001,2026-09-02T02:45:00Z,5
5,ORD-009718,CUST-00002,2026-06-29T21:48:00Z,1
6,ORD-002388,CUST-00002,2026-08-25T02:09:00Z,2
7,ORD-007417,CUST-00002,2026-08-25T20:39:00Z,3
8,ORD-004568,CUST-00002,2026-08-27T02:03:00Z,4
9,ORD-003715,CUST-00003,2026-06-24T15:01:00Z,1


In [10]:
customer_order_rank["new_customer_flag"] = (
    customer_order_rank["customer_order_rank"] == 1
)

customer_order_rank["repeat_customer_flag"] = (
    customer_order_rank["customer_order_rank"] > 1
)

print(
    "New customer orders:",
    customer_order_rank["new_customer_flag"].sum()
)

print(
    "Repeat customer orders:",
    customer_order_rank["repeat_customer_flag"].sum()
)

print(
    "Total:",
    customer_order_rank["new_customer_flag"].sum()
    + customer_order_rank["repeat_customer_flag"].sum()
)

display(customer_order_rank.head(10))

New customer orders: 2455
Repeat customer orders: 7545
Total: 10000


,order_id,customer_id,ordered_at_utc,customer_order_rank,new_customer_flag,repeat_customer_flag
0,ORD-006860,CUST-00001,2026-06-26T15:18:00Z,1,True,False
1,ORD-009732,CUST-00001,2026-07-18T01:09:00Z,2,False,True
2,ORD-002644,CUST-00001,2026-08-12T23:07:00Z,3,False,True
3,ORD-006550,CUST-00001,2026-08-19T06:24:00Z,4,False,True
4,ORD-009265,CUST-00001,2026-09-02T02:45:00Z,5,False,True
5,ORD-009718,CUST-00002,2026-06-29T21:48:00Z,1,True,False
6,ORD-002388,CUST-00002,2026-08-25T02:09:00Z,2,False,True
7,ORD-007417,CUST-00002,2026-08-25T20:39:00Z,3,False,True
8,ORD-004568,CUST-00002,2026-08-27T02:03:00Z,4,False,True
9,ORD-003715,CUST-00003,2026-06-24T15:01:00Z,1,True,False


In [11]:
customer_order_enriched = (
    customer_order_base
    .merge(
        refund_by_order,
        on="order_id",
        how="left"
    )
    .merge(
        return_by_order,
        on="order_id",
        how="left"
    )
)

# Order tanpa refund/return = 0
customer_order_enriched["refund_amount"] = (
    customer_order_enriched["refund_amount"]
    .fillna(0)
    .round(2)
)

customer_order_enriched["return_count"] = (
    customer_order_enriched["return_count"]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

print(
    "Orders with refund:",
    (customer_order_enriched["refund_amount"] > 0).sum()
)

print(
    "Orders with return:",
    (customer_order_enriched["return_count"] > 0).sum()
)

display(customer_order_enriched.head(10))

Rows: 10000
Unique orders: 10000
Orders with refund: 1991
Orders with return: 1372


,order_id,customer_id,metric_date,sales_channel,unit_quantity,gross_revenue,discount_amount,shipping_revenue,refund_amount,return_count
0,ORD-000299,CUST-00552,2026-07-23,MOBILE_APP,5.0,681.90,5.0,7.5,0.00,0
1,ORD-005343,CUST-00095,2026-08-12,WEB,4.0,268.04,5.0,7.5,0.00,0
2,ORD-005158,CUST-00734,2026-06-30,WEB,6.0,1372.00,2.5,7.5,0.00,1
3,ORD-002601,CUST-00167,2026-08-20,STORE,8.0,1557.97,7.5,3.5,776.99,0
4,ORD-006850,CUST-00518,2026-09-11,WEB,5.0,419.11,0.0,0.0,406.11,0
5,ORD-005256,CUST-01362,2026-08-21,STORE,1.0,298.58,0.0,0.0,0.00,0
6,ORD-003129,CUST-01529,2026-07-06,WEB,2.0,783.54,0.0,0.0,0.00,0
7,ORD-008070,CUST-00247,2026-06-30,WEB,6.0,1926.75,0.0,12.0,0.00,0
8,ORD-009333,CUST-01271,2026-07-08,MOBILE_APP,3.0,341.82,0.0,0.0,0.00,0
9,ORD-004295,CUST-02308,2026-07-17,MARKETPLACE,5.0,421.27,0.0,7.5,150.07,0


In [12]:
customer_order_enriched = customer_order_enriched.merge(
    customer_order_rank[
        [
            "order_id",
            "new_customer_flag",
            "repeat_customer_flag"
        ]
    ],
    on="order_id",
    how="left"
)

print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

print(
    "New customer orders:",
    customer_order_enriched["new_customer_flag"].sum()
)

print(
    "Repeat customer orders:",
    customer_order_enriched["repeat_customer_flag"].sum()
)

print("\nNULL flags:")
display(
    customer_order_enriched[
        ["new_customer_flag", "repeat_customer_flag"]
    ].isna().sum()
)

Rows: 10000
Unique orders: 10000
New customer orders: 2455
Repeat customer orders: 7545

NULL flags:


new_customer_flag       0
repeat_customer_flag    0
dtype: int64

In [13]:
customer_order_enriched["net_revenue"] = (
    customer_order_enriched["gross_revenue"]
    - customer_order_enriched["discount_amount"]
    + customer_order_enriched["shipping_revenue"]
    - customer_order_enriched["refund_amount"]
).round(2)

print("Net revenue selesai dihitung.")

display(
    customer_order_enriched[
        [
            "order_id",
            "gross_revenue",
            "discount_amount",
            "shipping_revenue",
            "refund_amount",
            "net_revenue"
        ]
    ].head(10)
)

Net revenue selesai dihitung.


,order_id,gross_revenue,discount_amount,shipping_revenue,refund_amount,net_revenue
0,ORD-000299,681.90,5.0,7.5,0.00,684.40
1,ORD-005343,268.04,5.0,7.5,0.00,270.54
2,ORD-005158,1372.00,2.5,7.5,0.00,1377.00
3,ORD-002601,1557.97,7.5,3.5,776.99,776.98
4,ORD-006850,419.11,0.0,0.0,406.11,13.00
5,ORD-005256,298.58,0.0,0.0,0.00,298.58
6,ORD-003129,783.54,0.0,0.0,0.00,783.54
7,ORD-008070,1926.75,0.0,12.0,0.00,1938.75
8,ORD-009333,341.82,0.0,0.0,0.00,341.82
9,ORD-004295,421.27,0.0,7.5,150.07,278.70


In [14]:
net_revenue_check = customer_order_enriched[
    customer_order_enriched["net_revenue"].round(2)
    != (
        customer_order_enriched["gross_revenue"]
        - customer_order_enriched["discount_amount"]
        + customer_order_enriched["shipping_revenue"]
        - customer_order_enriched["refund_amount"]
    ).round(2)
]

print("Net revenue mismatch:", len(net_revenue_check))

Net revenue mismatch: 0


In [15]:
print("=== ENRICHED DATA CHECK ===")

print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

print(
    "Unique customers:",
    customer_order_enriched["customer_id"].nunique()
)

print("\nNULL check:")
display(
    customer_order_enriched[
        [
            "order_id",
            "customer_id",
            "metric_date",
            "sales_channel",
            "unit_quantity",
            "gross_revenue",
            "discount_amount",
            "shipping_revenue",
            "refund_amount",
            "return_count",
            "new_customer_flag",
            "repeat_customer_flag",
            "net_revenue"
        ]
    ].isna().sum()
)

=== ENRICHED DATA CHECK ===
Rows: 10000
Unique orders: 10000
Unique customers: 2455

NULL check:


order_id                0
customer_id             0
metric_date             0
sales_channel           0
unit_quantity           0
gross_revenue           0
discount_amount         0
shipping_revenue        0
refund_amount           0
return_count            0
new_customer_flag       0
repeat_customer_flag    0
net_revenue             0
dtype: int64

In [16]:
negative_net_revenue = customer_order_enriched[
    customer_order_enriched["net_revenue"] < 0
].copy()

print(
    "Negative net revenue:",
    len(negative_net_revenue)
)

display(
    negative_net_revenue[
        [
            "order_id",
            "gross_revenue",
            "discount_amount",
            "shipping_revenue",
            "refund_amount",
            "net_revenue",
            "return_count"
        ]
    ]
    .sort_values("net_revenue")
    .head(20)
)

Negative net revenue: 0


,order_id,gross_revenue,discount_amount,shipping_revenue,refund_amount,net_revenue,return_count


In [19]:
payment_summary = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,

        MAX(
            CASE
                WHEN event_type = 'PAYMENT_CAPTURED'
                THEN "payload.amount"
                ELSE 0
            END
        ) AS captured_payment_amount,

        CASE
            WHEN COUNT(*) FILTER (
                WHERE event_type = 'PAYMENT_CAPTURED'
            ) > 0
                THEN 'PAYMENT_CAPTURED'

            WHEN COUNT(*) FILTER (
                WHERE event_type = 'PAYMENT_AUTHORIZED'
            ) > 0
                THEN 'PAYMENT_AUTHORIZED'

            WHEN COUNT(*) FILTER (
                WHERE event_type = 'PAYMENT_FAILED'
            ) > 0
                THEN 'PAYMENT_FAILED'

            ELSE 'UNKNOWN'
        END AS payment_status

    FROM silver.payment_events

    GROUP BY "payload.order_id"
    """,
    engine
)

print("Payment summary rows:", len(payment_summary))

display(payment_summary.head(10))

Payment summary rows: 10000


,order_id,captured_payment_amount,payment_status
0,ORD-000299,676.40,PAYMENT_CAPTURED
1,ORD-005343,270.54,PAYMENT_CAPTURED
2,ORD-002601,1553.97,PAYMENT_CAPTURED
3,ORD-005158,1352.00,PAYMENT_CAPTURED
4,ORD-006850,406.11,PAYMENT_CAPTURED
5,ORD-005256,292.58,PAYMENT_CAPTURED
6,ORD-003129,422.70,PAYMENT_CAPTURED
7,ORD-008070,1932.75,PAYMENT_CAPTURED
8,ORD-009333,341.82,PAYMENT_CAPTURED
9,ORD-004295,428.77,PAYMENT_CAPTURED


In [20]:
print("Unique payment orders:", payment_summary["order_id"].nunique())

display(
    payment_summary["payment_status"].value_counts()
)

Unique payment orders: 10000


payment_status
PAYMENT_CAPTURED      8785
PAYMENT_AUTHORIZED    1215
Name: count, dtype: int64

In [21]:
customer_order_enriched = customer_order_enriched.merge(
    payment_summary,
    on="order_id",
    how="left"
)

print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

print("\nNULL payment fields:")
display(
    customer_order_enriched[
        [
            "payment_status",
            "captured_payment_amount"
        ]
    ].isna().sum()
)

Rows: 10000
Unique orders: 10000

NULL payment fields:


payment_status             0
captured_payment_amount    0
dtype: int64

In [22]:
payment_check = customer_order_enriched[
    (
        (customer_order_enriched["payment_status"] == "PAYMENT_CAPTURED")
        & (customer_order_enriched["captured_payment_amount"] <= 0)
    )
    |
    (
        (customer_order_enriched["payment_status"] == "PAYMENT_AUTHORIZED")
        & (customer_order_enriched["captured_payment_amount"] > 0)
    )
].copy()

print("Payment inconsistency:", len(payment_check))

display(
    payment_check[
        [
            "order_id",
            "payment_status",
            "captured_payment_amount",
            "net_revenue"
        ]
    ].head(20)
)

Payment inconsistency: 9


,order_id,payment_status,captured_payment_amount,net_revenue
891,ORD-000511,PAYMENT_CAPTURED,0.0,17.59
1924,ORD-002406,PAYMENT_CAPTURED,0.0,14.81
2025,ORD-002279,PAYMENT_CAPTURED,0.0,11.48
3190,ORD-003652,PAYMENT_CAPTURED,0.0,11.48
6934,ORD-008159,PAYMENT_CAPTURED,0.0,11.48
7162,ORD-007706,PAYMENT_CAPTURED,0.0,22.96
7854,ORD-006568,PAYMENT_CAPTURED,0.0,11.48
8330,ORD-005053,PAYMENT_CAPTURED,0.0,14.81
9533,ORD-007666,PAYMENT_CAPTURED,0.0,14.98


In [23]:
payment_anomaly_ids = payment_check["order_id"].tolist()

payment_anomaly_detail = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        event_id,
        event_type,
        "payload.amount" AS payment_amount,
        occurred_at_utc
    FROM silver.payment_events
    WHERE "payload.order_id" = ANY(%(order_ids)s)
    ORDER BY "payload.order_id", occurred_at_utc
    """,
    engine,
    params={"order_ids": payment_anomaly_ids}
)

display(payment_anomaly_detail)

,order_id,event_id,event_type,payment_amount,occurred_at_utc
0,ORD-000511,EVT-00003107,PAYMENT_AUTHORIZED,0.0,2026-09-06T04:31:00Z
1,ORD-000511,EVT-00003108,PAYMENT_CAPTURED,0.0,2026-09-06T05:26:00Z
2,ORD-002279,EVT-00014077,PAYMENT_AUTHORIZED,0.0,2026-08-06T18:36:00Z
3,ORD-002279,EVT-00014078,PAYMENT_CAPTURED,0.0,2026-08-06T19:31:00Z
4,ORD-002406,EVT-00014897,PAYMENT_AUTHORIZED,0.0,2026-07-13T12:56:00Z
5,ORD-002406,EVT-00014898,PAYMENT_CAPTURED,0.0,2026-07-13T13:51:00Z
6,ORD-003652,EVT-00022658,PAYMENT_AUTHORIZED,0.0,2026-08-11T15:04:00Z
7,ORD-003652,EVT-00022659,PAYMENT_CAPTURED,0.0,2026-08-11T15:59:00Z
8,ORD-005053,EVT-00031365,PAYMENT_AUTHORIZED,0.0,2026-07-21T04:54:00Z
9,ORD-005053,EVT-00031366,PAYMENT_CAPTURED,0.0,2026-07-21T05:49:00Z


In [25]:
return_summary = pd.read_sql(
    """
    WITH return_events_ranked AS (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.return_events
    )

    SELECT
        order_id,
        event_type AS return_status
    FROM return_events_ranked
    WHERE rn = 1
    """,
    engine
)

print("Return summary rows:", len(return_summary))
print("Unique return orders:", return_summary["order_id"].nunique())

display(return_summary.head(10))

Return summary rows: 1372
Unique return orders: 1372


,order_id,return_status
0,ORD-000003,RETURN_REQUESTED
1,ORD-000011,RETURN_CLOSED
2,ORD-000016,RETURN_RECEIVED
3,ORD-000029,RETURN_CLOSED
4,ORD-000034,RETURN_RECEIVED
5,ORD-000039,RETURN_CLOSED
6,ORD-000043,RETURN_CLOSED
7,ORD-000044,RETURN_CLOSED
8,ORD-000049,RETURN_REQUESTED
9,ORD-000056,RETURN_REQUESTED


In [26]:
customer_order_enriched = customer_order_enriched.merge(
    return_summary,
    on="order_id",
    how="left"
)

customer_order_enriched["return_status"] = (
    customer_order_enriched["return_status"]
    .fillna("NO_RETURN")
)

print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

print("\nReturn status:")
display(
    customer_order_enriched["return_status"].value_counts()
)

print("\nNULL return status:")
print(
    customer_order_enriched["return_status"].isna().sum()
)

Rows: 10000
Unique orders: 10000

Return status:


return_status
NO_RETURN           8628
RETURN_CLOSED        947
RETURN_REQUESTED     251
RETURN_RECEIVED      174
Name: count, dtype: int64


NULL return status:
0


In [27]:
return_check = customer_order_enriched[
    (
        (customer_order_enriched["return_count"] == 0)
        & (customer_order_enriched["return_status"] != "NO_RETURN")
    )
    |
    (
        (customer_order_enriched["return_count"] > 0)
        & (customer_order_enriched["return_status"] == "NO_RETURN")
    )
].copy()

print(
    "Return status inconsistency:",
    len(return_check)
)

display(
    return_check[
        [
            "order_id",
            "return_count",
            "return_status",
            "refund_amount",
            "net_revenue"
        ]
    ].head(20)
)

Return status inconsistency: 0


,order_id,return_count,return_status,refund_amount,net_revenue


In [28]:
print("=== REFUND VALIDATION ===")

print(
    "Orders with refund:",
    (customer_order_enriched["refund_amount"] > 0).sum()
)

print(
    "Orders without refund:",
    (customer_order_enriched["refund_amount"] == 0).sum()
)

print(
    "Negative refund:",
    (customer_order_enriched["refund_amount"] < 0).sum()
)

print(
    "Total refund:",
    round(customer_order_enriched["refund_amount"].sum(), 2)
)

=== REFUND VALIDATION ===
Orders with refund: 1991
Orders without refund: 8009
Negative refund: 0
Total refund: 1549240.21


In [29]:
silver_refund_total = pd.read_sql(
    """
    SELECT
        SUM("payload.amount") AS total_refund
    FROM silver.refund_events
    WHERE event_type = 'REFUND_COMPLETED'
    """,
    engine
).iloc[0]["total_refund"]

gold_refund_total = customer_order_enriched["refund_amount"].sum()

print("Silver refund total:", round(float(silver_refund_total), 2))
print("Gold refund total:", round(float(gold_refund_total), 2))
print(
    "Difference:",
    round(float(silver_refund_total) - float(gold_refund_total), 2)
)

Silver refund total: 1606522.99
Gold refund total: 1549240.21
Difference: 57282.78


In [30]:
refund_duplicate_check = pd.read_sql(
    """
    SELECT
        "payload.refund_id" AS refund_id,
        "payload.order_id" AS order_id,
        event_type,
        "payload.amount" AS amount,
        occurred_at_utc,
        COUNT(*) AS duplicate_count
    FROM silver.refund_events
    WHERE event_type = 'REFUND_COMPLETED'
    GROUP BY
        "payload.refund_id",
        "payload.order_id",
        event_type,
        "payload.amount",
        occurred_at_utc
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC
    """,
    engine
)

print(
    "Duplicate refund events:",
    len(refund_duplicate_check)
)

display(refund_duplicate_check.head(20))

Duplicate refund events: 73


,refund_id,order_id,event_type,amount,occurred_at_utc,duplicate_count
0,REF-ORD-005344-01,ORD-005344,REFUND_COMPLETED,1429.76,2026-07-19T01:13:00Z,2
1,REF-ORD-002903-01,ORD-002903,REFUND_COMPLETED,220.74,2026-07-04T14:43:00Z,2
2,REF-ORD-009599-01,ORD-009599,REFUND_COMPLETED,1135.55,2026-09-20T06:24:00+07:00,2
3,REF-ORD-002717-01,ORD-002717,REFUND_COMPLETED,361.89,2026-07-13T11:32:00Z,2
4,REF-ORD-000950-01,ORD-000950,REFUND_COMPLETED,217.59,2026-08-10T15:45:00Z,2
5,REF-ORD-009189-01,ORD-009189,REFUND_COMPLETED,713.31,2026-07-10T19:20:00Z,2
6,REF-ORD-004265-01,ORD-004265,REFUND_COMPLETED,2108.42,2026-06-27T22:23:00Z,2
7,REF-ORD-000748-01,ORD-000748,REFUND_COMPLETED,3245.06,2026-07-05T01:16:00Z,2
8,REF-ORD-005044-01,ORD-005044,REFUND_COMPLETED,434.34,2026-08-29T00:02:00Z,2
9,REF-ORD-002479-01,ORD-002479,REFUND_COMPLETED,945.27,2026-07-28T20:23:00+07:00,2


In [31]:
refund_duplicate_impact = pd.read_sql(
    """
    WITH duplicates AS (
        SELECT
            "payload.refund_id" AS refund_id,
            "payload.order_id" AS order_id,
            "payload.amount" AS amount,
            occurred_at_utc,
            COUNT(*) AS duplicate_count
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY
            "payload.refund_id",
            "payload.order_id",
            "payload.amount",
            occurred_at_utc
        HAVING COUNT(*) > 1
    )

    SELECT
        SUM(amount * (duplicate_count - 1)) AS duplicate_amount
    FROM duplicates
    """,
    engine
)

display(refund_duplicate_impact)

,duplicate_amount
0,57282.78


In [32]:
refund_reconciliation = pd.read_sql(
    """
    WITH refund_dedup AS (
        SELECT
            "payload.refund_id" AS refund_id,
            "payload.order_id" AS order_id,
            "payload.amount" AS amount,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.refund_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
    )

    SELECT
        SUM(amount) AS deduplicated_refund_total
    FROM refund_dedup
    WHERE rn = 1
    """,
    engine
)

deduplicated_silver_refund = refund_reconciliation.iloc[0][
    "deduplicated_refund_total"
]

gold_refund_total = customer_order_enriched["refund_amount"].sum()

print(
    "Deduplicated Silver refund:",
    round(float(deduplicated_silver_refund), 2)
)

print(
    "Gold refund:",
    round(float(gold_refund_total), 2)
)

print(
    "Difference:",
    round(
        float(deduplicated_silver_refund) -
        float(gold_refund_total),
        2
    )
)

Deduplicated Silver refund: 1549240.21
Gold refund: 1549240.21
Difference: 0.0


In [34]:
customer_flag_check = customer_order_enriched[
    (
        (customer_order_enriched["new_customer_flag"] == True)
        & (customer_order_enriched["repeat_customer_flag"] == True)
    )
    |
    (
        (customer_order_enriched["new_customer_flag"] == False)
        & (customer_order_enriched["repeat_customer_flag"] == False)
    )
].copy()

print(
    "Customer flag mismatch:",
    len(customer_flag_check)
)

display(
    customer_flag_check[
        [
            "order_id",
            "customer_id",
            "new_customer_flag",
            "repeat_customer_flag"
        ]
    ].head(20)
)

Customer flag mismatch: 0


,order_id,customer_id,new_customer_flag,repeat_customer_flag


In [35]:
print("=== FINAL GOLD STRUCTURE CHECK ===")

print("Rows:", len(customer_order_enriched))
print("Columns:", len(customer_order_enriched.columns))

print("\nColumns:")
for i, col in enumerate(customer_order_enriched.columns, start=1):
    print(f"{i}. {col}")

=== FINAL GOLD STRUCTURE CHECK ===
Rows: 10000
Columns: 16

Columns:
1. order_id
2. customer_id
3. metric_date
4. sales_channel
5. unit_quantity
6. gross_revenue
7. discount_amount
8. shipping_revenue
9. refund_amount
10. return_count
11. new_customer_flag
12. repeat_customer_flag
13. net_revenue
14. captured_payment_amount
15. payment_status
16. return_status


In [36]:
print("=== FINAL DATA TYPE CHECK ===")

display(
    customer_order_enriched.dtypes.to_frame("data_type")
)

=== FINAL DATA TYPE CHECK ===


,data_type
order_id,object
customer_id,object
metric_date,object
sales_channel,object
unit_quantity,float64
gross_revenue,float64
discount_amount,float64
shipping_revenue,float64
refund_amount,float64
return_count,int32


In [37]:
print("=== FINAL NULL CHECK ===")

final_null_check = customer_order_enriched.isna().sum()

display(
    final_null_check[final_null_check > 0]
)

print(
    "\nTotal NULL:",
    final_null_check.sum()
)

=== FINAL NULL CHECK ===


Series([], dtype: int64)


Total NULL: 0


In [38]:
# ==========================================
# FINAL GOLD DATA TYPE PREPARATION
# ==========================================

customer_order_enriched["order_id"] = (
    customer_order_enriched["order_id"].astype(str)
)

customer_order_enriched["customer_id"] = (
    customer_order_enriched["customer_id"].astype(str)
)

customer_order_enriched["metric_date"] = pd.to_datetime(
    customer_order_enriched["metric_date"]
).dt.date

customer_order_enriched["sales_channel"] = (
    customer_order_enriched["sales_channel"].astype(str)
)

customer_order_enriched["unit_quantity"] = (
    customer_order_enriched["unit_quantity"].astype(int)
)

customer_order_enriched["gross_revenue"] = (
    customer_order_enriched["gross_revenue"].astype(float)
)

customer_order_enriched["discount_amount"] = (
    customer_order_enriched["discount_amount"].astype(float)
)

customer_order_enriched["shipping_revenue"] = (
    customer_order_enriched["shipping_revenue"].astype(float)
)

customer_order_enriched["refund_amount"] = (
    customer_order_enriched["refund_amount"].astype(float)
)

customer_order_enriched["return_count"] = (
    customer_order_enriched["return_count"].astype(int)
)

customer_order_enriched["new_customer_flag"] = (
    customer_order_enriched["new_customer_flag"].astype(bool)
)

customer_order_enriched["repeat_customer_flag"] = (
    customer_order_enriched["repeat_customer_flag"].astype(bool)
)

customer_order_enriched["net_revenue"] = (
    customer_order_enriched["net_revenue"].astype(float)
)

customer_order_enriched["captured_payment_amount"] = (
    customer_order_enriched["captured_payment_amount"].astype(float)
)

customer_order_enriched["payment_status"] = (
    customer_order_enriched["payment_status"].astype(str)
)

customer_order_enriched["return_status"] = (
    customer_order_enriched["return_status"].astype(str)
)

print("Final data type preparation selesai.")
display(customer_order_enriched.dtypes.to_frame("data_type"))

Final data type preparation selesai.


,data_type
order_id,object
customer_id,object
metric_date,object
sales_channel,object
unit_quantity,int32
gross_revenue,float64
discount_amount,float64
shipping_revenue,float64
refund_amount,float64
return_count,int32


In [39]:
# ==========================================
# FINAL GOLD BUSINESS SUMMARY
# ==========================================

print("=== FINAL GOLD BUSINESS SUMMARY ===")

print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

print(
    "Unique customers:",
    customer_order_enriched["customer_id"].nunique()
)

print(
    "New customer orders:",
    customer_order_enriched["new_customer_flag"].sum()
)

print(
    "Repeat customer orders:",
    customer_order_enriched["repeat_customer_flag"].sum()
)

print(
    "Orders with refund:",
    (customer_order_enriched["refund_amount"] > 0).sum()
)

print(
    "Orders with return:",
    (customer_order_enriched["return_count"] > 0).sum()
)

print(
    "Total gross revenue:",
    round(customer_order_enriched["gross_revenue"].sum(), 2)
)

print(
    "Total discount:",
    round(customer_order_enriched["discount_amount"].sum(), 2)
)

print(
    "Total shipping revenue:",
    round(customer_order_enriched["shipping_revenue"].sum(), 2)
)

print(
    "Total refund:",
    round(customer_order_enriched["refund_amount"].sum(), 2)
)

print(
    "Total net revenue:",
    round(customer_order_enriched["net_revenue"].sum(), 2)
)

print(
    "Total captured payment:",
    round(
        customer_order_enriched["captured_payment_amount"].sum(),
        2
    )
)

=== FINAL GOLD BUSINESS SUMMARY ===
Rows: 10000
Unique orders: 10000
Unique customers: 2455
New customer orders: 2455
Repeat customer orders: 7545
Orders with refund: 1991
Orders with return: 1372
Total gross revenue: 15102441.63
Total discount: 37177.5
Total shipping revenue: 44897.5
Total refund: 1549240.21
Total net revenue: 13560921.42
Total captured payment: 11815776.97


In [41]:
# ==========================================
# BUILD CUSTOMER DAILY
# Grain: 1 customer × 1 date
# ==========================================

customer_daily = (
    customer_order_enriched
    .groupby(
        ["customer_id", "metric_date"],
        as_index=False
    )
    .agg(
        order_count=("order_id", "nunique"),
        unit_quantity=("unit_quantity", "sum"),
        gross_revenue=("gross_revenue", "sum"),
        discount_amount=("discount_amount", "sum"),
        shipping_revenue=("shipping_revenue", "sum"),
        refund_amount=("refund_amount", "sum"),
        return_count=("return_count", "sum"),
        net_revenue=("net_revenue", "sum"),
        captured_payment_amount=("captured_payment_amount", "sum"),
        new_customer_order_count=("new_customer_flag", "sum"),
        repeat_customer_order_count=("repeat_customer_flag", "sum")
    )
)

# Rounding untuk metric keuangan
money_columns = [
    "gross_revenue",
    "discount_amount",
    "shipping_revenue",
    "refund_amount",
    "net_revenue",
    "captured_payment_amount"
]

customer_daily[money_columns] = (
    customer_daily[money_columns].round(2)
)

print("Customer daily berhasil dibuat.")
print("Rows:", len(customer_daily))
print("Unique customers:", customer_daily["customer_id"].nunique())
print(
    "Unique customer-date:",
    customer_daily[["customer_id", "metric_date"]]
    .drop_duplicates()
    .shape[0]
)

display(customer_daily.head(10))

Customer daily berhasil dibuat.
Rows: 9790
Unique customers: 2455
Unique customer-date: 9790


,customer_id,metric_date,order_count,unit_quantity,gross_revenue,discount_amount,shipping_revenue,refund_amount,return_count,net_revenue,captured_payment_amount,new_customer_order_count,repeat_customer_order_count
0,CUST-00001,2026-06-26,1,9,2020.63,5.0,0.0,997.82,1,1017.81,1995.63,1,0
1,CUST-00001,2026-07-18,1,5,548.59,0.0,0.0,0.00,1,548.59,533.59,0,1
2,CUST-00001,2026-08-13,1,7,1818.62,5.0,0.0,0.00,0,1813.62,1798.62,0,1
3,CUST-00001,2026-08-19,1,11,2593.71,0.0,3.5,0.00,0,2597.21,1424.07,0,1
4,CUST-00001,2026-09-02,1,13,3227.17,5.0,12.0,0.00,0,3234.17,0.00,0,1
5,CUST-00002,2026-06-30,1,7,2378.59,7.5,0.0,2351.09,0,20.00,2351.09,1,0
6,CUST-00002,2026-08-25,1,4,489.98,2.5,0.0,0.00,0,487.48,462.48,0,1
7,CUST-00002,2026-08-26,1,10,1402.24,5.0,0.0,0.00,1,1397.24,1384.24,0,1
8,CUST-00002,2026-08-27,1,2,194.56,0.0,7.5,64.42,0,137.64,184.06,0,1
9,CUST-00003,2026-06-24,1,2,608.72,0.0,3.5,0.00,0,612.22,0.00,1,0


In [42]:
# ==========================================
# CUSTOMER DAILY RECONCILIATION
# ==========================================

print("=== CUSTOMER DAILY RECONCILIATION ===")

print(
    "Order count - source:",
    len(customer_order_enriched)
)

print(
    "Order count - daily:",
    customer_daily["order_count"].sum()
)

print()

print(
    "Gross revenue - source:",
    round(
        customer_order_enriched["gross_revenue"].sum(),
        2
    )
)

print(
    "Gross revenue - daily:",
    round(
        customer_daily["gross_revenue"].sum(),
        2
    )
)

print()

print(
    "Discount - source:",
    round(
        customer_order_enriched["discount_amount"].sum(),
        2
    )
)

print(
    "Discount - daily:",
    round(
        customer_daily["discount_amount"].sum(),
        2
    )
)

print()

print(
    "Refund - source:",
    round(
        customer_order_enriched["refund_amount"].sum(),
        2
    )
)

print(
    "Refund - daily:",
    round(
        customer_daily["refund_amount"].sum(),
        2
    )
)

print()

print(
    "Net revenue - source:",
    round(
        customer_order_enriched["net_revenue"].sum(),
        2
    )
)

print(
    "Net revenue - daily:",
    round(
        customer_daily["net_revenue"].sum(),
        2
    )
)

=== CUSTOMER DAILY RECONCILIATION ===
Order count - source: 10000
Order count - daily: 10000

Gross revenue - source: 15102441.63
Gross revenue - daily: 15102441.63

Discount - source: 37177.5
Discount - daily: 37177.5

Refund - source: 1549240.21
Refund - daily: 1549240.21

Net revenue - source: 13560921.42
Net revenue - daily: 13560921.42


In [43]:
# ==========================================
# CHECK MULTIPLE ORDERS IN SAME CUSTOMER-DATE
# ==========================================

multi_order_days = customer_daily[
    customer_daily["order_count"] > 1
].copy()

print(
    "Customer-date dengan >1 order:",
    len(multi_order_days)
)

print(
    "Order yang berada dalam customer-date tersebut:",
    multi_order_days["order_count"].sum()
)

display(
    multi_order_days.head(20)
)

Customer-date dengan >1 order: 206
Order yang berada dalam customer-date tersebut: 416


,customer_id,metric_date,order_count,unit_quantity,gross_revenue,discount_amount,shipping_revenue,refund_amount,return_count,net_revenue,captured_payment_amount,new_customer_order_count,repeat_customer_order_count
82,CUST-00021,2026-07-11,2,16,3260.71,12.5,12.0,1928.10,0,1332.11,2648.11,1,1
91,CUST-00024,2026-07-26,2,15,5274.30,0.0,12.0,0.00,0,5286.30,3873.92,0,2
109,CUST-00030,2026-07-03,2,12,3046.95,10.0,3.5,2574.33,0,466.12,3002.45,1,1
141,CUST-00036,2026-07-18,2,14,3024.03,10.0,0.0,1131.07,1,1882.96,1895.18,0,2
175,CUST-00043,2026-08-25,2,17,2933.78,2.5,3.5,0.00,0,2934.78,2904.78,0,2
226,CUST-00059,2026-07-19,2,14,4273.90,7.5,11.0,618.03,0,3659.37,633.74,0,2
385,CUST-00100,2026-08-06,2,13,3364.92,7.5,19.5,0.00,1,3376.92,509.85,0,2
441,CUST-00115,2026-06-27,2,8,1605.11,5.0,19.5,0.00,0,1619.61,1009.09,1,1
450,CUST-00117,2026-07-09,2,7,1151.59,5.0,12.0,0.00,1,1158.59,1110.76,0,2
594,CUST-00154,2026-07-06,2,13,2600.41,7.5,19.5,63.30,1,2549.11,1451.56,0,2


In [44]:
# ==========================================
# CUSTOMER DAILY BUSINESS VALIDATION
# ==========================================

print("=== CUSTOMER DAILY BUSINESS VALIDATION ===")

# 1. Order count harus > 0
print(
    "Order count <= 0:",
    (customer_daily["order_count"] <= 0).sum()
)

# 2. Revenue negatif
print(
    "Negative gross revenue:",
    (customer_daily["gross_revenue"] < 0).sum()
)

print(
    "Negative net revenue:",
    (customer_daily["net_revenue"] < 0).sum()
)

# 3. Refund negatif
print(
    "Negative refund:",
    (customer_daily["refund_amount"] < 0).sum()
)

# 4. Return count tidak boleh negatif
print(
    "Negative return count:",
    (customer_daily["return_count"] < 0).sum()
)

# 5. Customer flag consistency
flag_check = customer_daily[
    (
        customer_daily["new_customer_order_count"]
        + customer_daily["repeat_customer_order_count"]
    )
    != customer_daily["order_count"]
]

print(
    "Customer flag mismatch:",
    len(flag_check)
)

# 6. NULL check
null_check = customer_daily.isna().sum()

print("\nNULL check:")
display(null_check[null_check > 0])

=== CUSTOMER DAILY BUSINESS VALIDATION ===
Order count <= 0: 0
Negative gross revenue: 0
Negative net revenue: 0
Negative refund: 0
Negative return count: 0
Customer flag mismatch: 0

NULL check:


Series([], dtype: int64)

In [45]:
# ==========================================
# FINAL CUSTOMER DAILY STRUCTURE
# ==========================================

print("=== FINAL CUSTOMER DAILY STRUCTURE ===")

print("Rows:", len(customer_daily))
print("Columns:", len(customer_daily.columns))

print("\nColumns:")
for i, col in enumerate(customer_daily.columns, start=1):
    print(f"{i}. {col}")

print("\nData types:")
display(
    customer_daily.dtypes.to_frame("data_type")
)

print("\nSample:")
display(customer_daily.head(10))

=== FINAL CUSTOMER DAILY STRUCTURE ===
Rows: 9790
Columns: 13

Columns:
1. customer_id
2. metric_date
3. order_count
4. unit_quantity
5. gross_revenue
6. discount_amount
7. shipping_revenue
8. refund_amount
9. return_count
10. net_revenue
11. captured_payment_amount
12. new_customer_order_count
13. repeat_customer_order_count

Data types:


,data_type
customer_id,object
metric_date,object
order_count,int64
unit_quantity,int32
gross_revenue,float64
discount_amount,float64
shipping_revenue,float64
refund_amount,float64
return_count,int32
net_revenue,float64



Sample:


,customer_id,metric_date,order_count,unit_quantity,gross_revenue,discount_amount,shipping_revenue,refund_amount,return_count,net_revenue,captured_payment_amount,new_customer_order_count,repeat_customer_order_count
0,CUST-00001,2026-06-26,1,9,2020.63,5.0,0.0,997.82,1,1017.81,1995.63,1,0
1,CUST-00001,2026-07-18,1,5,548.59,0.0,0.0,0.00,1,548.59,533.59,0,1
2,CUST-00001,2026-08-13,1,7,1818.62,5.0,0.0,0.00,0,1813.62,1798.62,0,1
3,CUST-00001,2026-08-19,1,11,2593.71,0.0,3.5,0.00,0,2597.21,1424.07,0,1
4,CUST-00001,2026-09-02,1,13,3227.17,5.0,12.0,0.00,0,3234.17,0.00,0,1
5,CUST-00002,2026-06-30,1,7,2378.59,7.5,0.0,2351.09,0,20.00,2351.09,1,0
6,CUST-00002,2026-08-25,1,4,489.98,2.5,0.0,0.00,0,487.48,462.48,0,1
7,CUST-00002,2026-08-26,1,10,1402.24,5.0,0.0,0.00,1,1397.24,1384.24,0,1
8,CUST-00002,2026-08-27,1,2,194.56,0.0,7.5,64.42,0,137.64,184.06,0,1
9,CUST-00003,2026-06-24,1,2,608.72,0.0,3.5,0.00,0,612.22,0.00,1,0


In [46]:
# ==========================================
# FINAL DATA TYPE PREPARATION
# ==========================================

customer_daily["metric_date"] = pd.to_datetime(
    customer_daily["metric_date"]
).dt.date

customer_daily["order_count"] = (
    customer_daily["order_count"].astype("int32")
)

customer_daily["unit_quantity"] = (
    customer_daily["unit_quantity"].astype("int32")
)

customer_daily["return_count"] = (
    customer_daily["return_count"].astype("int32")
)

customer_daily["new_customer_order_count"] = (
    customer_daily["new_customer_order_count"].astype("int32")
)

customer_daily["repeat_customer_order_count"] = (
    customer_daily["repeat_customer_order_count"].astype("int32")
)

print("Final data type preparation selesai.")

display(
    customer_daily.dtypes.to_frame("data_type")
)

Final data type preparation selesai.


,data_type
customer_id,object
metric_date,object
order_count,int32
unit_quantity,int32
gross_revenue,float64
discount_amount,float64
shipping_revenue,float64
refund_amount,float64
return_count,int32
net_revenue,float64


In [47]:
from datetime import datetime, timezone

pipeline_run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

customer_daily["pipeline_run_id"] = pipeline_run_id

print("Pipeline run ID:", pipeline_run_id)
print("Rows:", len(customer_daily))
print("Columns:", len(customer_daily.columns))

Pipeline run ID: 20260924035304
Rows: 9790
Columns: 14


In [49]:
from sqlalchemy import text

create_customer_daily_sql = """
CREATE SCHEMA IF NOT EXISTS gold;

DROP TABLE IF EXISTS gold.customer_daily;

CREATE TABLE gold.customer_daily (
    customer_id TEXT NOT NULL,
    metric_date DATE NOT NULL,

    order_count INTEGER NOT NULL,
    unit_quantity INTEGER NOT NULL,

    gross_revenue NUMERIC(14, 2) NOT NULL,
    discount_amount NUMERIC(14, 2) NOT NULL,
    shipping_revenue NUMERIC(14, 2) NOT NULL,
    refund_amount NUMERIC(14, 2) NOT NULL,

    return_count INTEGER NOT NULL,

    net_revenue NUMERIC(14, 2) NOT NULL,
    captured_payment_amount NUMERIC(14, 2) NOT NULL,

    new_customer_order_count INTEGER NOT NULL,
    repeat_customer_order_count INTEGER NOT NULL,

    pipeline_run_id TEXT NOT NULL,

    PRIMARY KEY (customer_id, metric_date)
);
"""

with engine.begin() as conn:
    conn.execute(text(create_customer_daily_sql))

print("gold.customer_daily berhasil dibuat.")

gold.customer_daily berhasil dibuat.


In [50]:
# ==========================================
# CHECK GOLD CUSTOMER DAILY SCHEMA
# ==========================================

customer_daily_schema = pd.read_sql(
    """
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'gold'
      AND table_name = 'customer_daily'
    ORDER BY ordinal_position
    """,
    engine
)

display(customer_daily_schema)

,column_name,data_type,is_nullable
0,customer_id,text,NO
1,metric_date,date,NO
2,order_count,integer,NO
3,unit_quantity,integer,NO
4,gross_revenue,numeric,NO
5,discount_amount,numeric,NO
6,shipping_revenue,numeric,NO
7,refund_amount,numeric,NO
8,return_count,integer,NO
9,net_revenue,numeric,NO


In [51]:
# ==========================================
# LOAD CUSTOMER DAILY TO GOLD
# ==========================================

customer_daily.to_sql(
    "customer_daily",
    engine,
    schema="gold",
    if_exists="append",
    index=False,
    method="multi"
)

print("Data berhasil dimasukkan ke gold.customer_daily.")
print("Rows inserted:", len(customer_daily))

Data berhasil dimasukkan ke gold.customer_daily.
Rows inserted: 9790


In [52]:
# ==========================================
# FINAL GOLD CUSTOMER DAILY CHECK
# ==========================================

gold_customer_daily_check = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT (customer_id, metric_date)) AS unique_customer_dates,
        COUNT(*) - COUNT(DISTINCT (customer_id, metric_date)) AS duplicate_rows,
        COUNT(*) FILTER (
            WHERE customer_id IS NULL
               OR metric_date IS NULL
               OR order_count IS NULL
               OR gross_revenue IS NULL
               OR net_revenue IS NULL
        ) AS null_rows
    FROM gold.customer_daily
    """,
    engine
)

display(gold_customer_daily_check)

,total_rows,unique_customers,unique_customer_dates,duplicate_rows,null_rows
0,9790,2455,9790,0,0


In [53]:
# ==========================================
# FINAL GOLD RECONCILIATION
# ==========================================

gold_reconciliation = pd.read_sql(
    """
    SELECT
        o.total_orders,
        c.total_customer_daily_rows,

        o.total_gross_revenue,
        c.total_gross_revenue_daily,

        o.total_discount,
        c.total_discount_daily,

        o.total_shipping_revenue,
        c.total_shipping_revenue_daily,

        o.total_refund,
        c.total_refund_daily,

        o.total_net_revenue,
        c.total_net_revenue_daily,

        o.total_captured_payment,
        c.total_captured_payment_daily

    FROM
        (
            SELECT
                COUNT(*) AS total_orders,
                SUM(gross_merchandise_value) AS total_gross_revenue,
                SUM(discount_amount) AS total_discount,
                SUM(shipping_revenue) AS total_shipping_revenue,
                SUM(refunded_amount) AS total_refund,
                SUM(net_revenue) AS total_net_revenue,
                SUM(captured_payment_amount) AS total_captured_payment
            FROM gold.order_360
        ) o

    CROSS JOIN
        (
            SELECT
                COUNT(*) AS total_customer_daily_rows,
                SUM(gross_revenue) AS total_gross_revenue_daily,
                SUM(discount_amount) AS total_discount_daily,
                SUM(shipping_revenue) AS total_shipping_revenue_daily,
                SUM(refund_amount) AS total_refund_daily,
                SUM(net_revenue) AS total_net_revenue_daily,
                SUM(captured_payment_amount) AS total_captured_payment_daily
            FROM gold.customer_daily
        ) c
    """,
    engine
)

display(gold_reconciliation)

,total_orders,total_customer_daily_rows,total_gross_revenue,total_gross_revenue_daily,total_discount,total_discount_daily,total_shipping_revenue,total_shipping_revenue_daily,total_refund,total_refund_daily,total_net_revenue,total_net_revenue_daily,total_captured_payment,total_captured_payment_daily
0,10000,9790,15102441.63,15102441.63,37177.5,37177.5,44897.5,44897.5,1549240.21,1549240.21,13560921.42,13560921.42,13535356.5,11815776.97


In [54]:
payment_diff_check = pd.read_sql(
    """
    SELECT
        o.order_id,
        o.captured_payment_amount AS order_360_payment,
        c.captured_payment_amount AS customer_daily_payment,
        ROUND(
            o.captured_payment_amount
            - COALESCE(c.captured_payment_amount, 0),
            2
        ) AS difference
    FROM gold.order_360 o
    LEFT JOIN gold.customer_daily c
        ON o.customer_id = c.customer_id
       AND o.order_date = c.metric_date
    WHERE ROUND(
        o.captured_payment_amount
        - COALESCE(c.captured_payment_amount, 0),
        2
    ) <> 0
    ORDER BY ABS(
        o.captured_payment_amount
        - COALESCE(c.captured_payment_amount, 0)
    ) DESC
    """,
    engine
)

print("Payment mismatch rows:", len(payment_diff_check))

display(payment_diff_check.head(20))

Payment mismatch rows: 4328


,order_id,order_360_payment,customer_daily_payment,difference
0,ORD-001091,8493.14,NaN,8493.14
1,ORD-007461,8233.12,NaN,8233.12
2,ORD-001662,7469.28,NaN,7469.28
3,ORD-008918,6897.48,NaN,6897.48
4,ORD-006126,5909.89,NaN,5909.89
5,ORD-001499,5801.64,NaN,5801.64
6,ORD-002230,5632.06,NaN,5632.06
7,ORD-007827,5548.52,NaN,5548.52
8,ORD-001302,5214.38,NaN,5214.38
9,ORD-005260,5238.33,104.36,5133.97


In [55]:
payment_reconciliation = pd.read_sql(
    """
    WITH order_daily AS (
        SELECT
            customer_id,
            order_date AS metric_date,
            SUM(captured_payment_amount) AS captured_payment_amount
        FROM gold.order_360
        GROUP BY customer_id, order_date
    )

    SELECT
        COALESCE(o.customer_id, c.customer_id) AS customer_id,
        COALESCE(o.metric_date, c.metric_date) AS metric_date,

        ROUND(
            COALESCE(o.captured_payment_amount, 0),
            2
        ) AS order_360_payment,

        ROUND(
            COALESCE(c.captured_payment_amount, 0),
            2
        ) AS customer_daily_payment,

        ROUND(
            COALESCE(o.captured_payment_amount, 0)
            - COALESCE(c.captured_payment_amount, 0),
            2
        ) AS difference

    FROM order_daily o
    FULL OUTER JOIN gold.customer_daily c
        ON o.customer_id = c.customer_id
       AND o.metric_date = c.metric_date

    WHERE ROUND(
        COALESCE(o.captured_payment_amount, 0)
        - COALESCE(c.captured_payment_amount, 0),
        2
    ) <> 0

    ORDER BY ABS(
        COALESCE(o.captured_payment_amount, 0)
        - COALESCE(c.captured_payment_amount, 0)
    ) DESC
    """,
    engine
)

print("Customer-date payment mismatches:", len(payment_reconciliation))

display(payment_reconciliation.head(20))

Customer-date payment mismatches: 6599


,customer_id,metric_date,order_360_payment,customer_daily_payment,difference
0,CUST-00112,2026-07-09,8493.14,0.00,8493.14
1,CUST-01576,2026-08-14,8233.12,0.00,8233.12
2,CUST-01444,2026-09-13,7469.28,0.00,7469.28
3,CUST-00157,2026-06-27,6897.48,0.00,6897.48
4,CUST-01243,2026-07-18,5909.89,0.00,5909.89
5,CUST-01617,2026-09-16,5801.64,0.00,5801.64
6,CUST-01117,2026-07-26,5632.06,0.00,5632.06
7,CUST-01117,2026-07-27,0.00,5632.06,-5632.06
8,CUST-02269,2026-08-15,0.00,5548.52,-5548.52
9,CUST-02269,2026-08-14,5548.52,0.00,5548.52


In [56]:
# ==========================================
# PAYMENT DATE RECONCILIATION
# ==========================================

payment_by_date = pd.read_sql(
    """
    WITH order_daily AS (
        SELECT
            order_date AS metric_date,
            SUM(captured_payment_amount) AS order_360_payment
        FROM gold.order_360
        GROUP BY order_date
    ),

    customer_daily AS (
        SELECT
            metric_date,
            SUM(captured_payment_amount) AS customer_daily_payment
        FROM gold.customer_daily
        GROUP BY metric_date
    )

    SELECT
        COALESCE(o.metric_date, c.metric_date) AS metric_date,

        ROUND(
            COALESCE(o.order_360_payment, 0),
            2
        ) AS order_360_payment,

        ROUND(
            COALESCE(c.customer_daily_payment, 0),
            2
        ) AS customer_daily_payment,

        ROUND(
            COALESCE(o.order_360_payment, 0)
            - COALESCE(c.customer_daily_payment, 0),
            2
        ) AS difference

    FROM order_daily o

    FULL OUTER JOIN customer_daily c
        ON o.metric_date = c.metric_date

    ORDER BY metric_date
    """,
    engine
)

display(payment_by_date)

,metric_date,order_360_payment,customer_daily_payment,difference
0,2026-06-22,149117.93,94376.55,54741.38
1,2026-06-23,154386.82,139143.13,15243.69
2,2026-06-24,128327.32,101195.95,27131.37
3,2026-06-25,138980.24,130808.32,8171.92
4,2026-06-26,135909.12,115541.98,20367.14
...,...,...,...,...
86,2026-09-16,139654.84,130410.70,9244.14
87,2026-09-17,143912.53,117404.90,26507.63
88,2026-09-18,142621.35,127350.72,15270.63
89,2026-09-19,132133.76,128317.35,3816.41


In [57]:
# ==========================================
# ORDER 360 vs ENRICHED PAYMENT CHECK
# ==========================================

gold_order_payment = pd.read_sql(
    """
    SELECT
        order_id,
        captured_payment_amount
    FROM gold.order_360
    """,
    engine
)

enriched_payment = customer_order_enriched[
    [
        "order_id",
        "captured_payment_amount"
    ]
].copy()

payment_compare = enriched_payment.merge(
    gold_order_payment,
    on="order_id",
    how="outer",
    suffixes=("_enriched", "_gold")
)

payment_compare["difference"] = (
    payment_compare["captured_payment_amount_enriched"].fillna(0)
    - payment_compare["captured_payment_amount_gold"].fillna(0)
).round(2)

payment_mismatch = payment_compare[
    payment_compare["difference"] != 0
].copy()

print("Total orders:", len(payment_compare))
print("Payment mismatches:", len(payment_mismatch))

display(
    payment_mismatch.sort_values(
        "difference",
        key=abs,
        ascending=False
    ).head(20)
)

Total orders: 10000
Payment mismatches: 2115


,order_id,captured_payment_amount_enriched,captured_payment_amount_gold,difference
5164,ORD-005165,2597.16,6847.04,-4249.88
1090,ORD-001091,4246.57,8493.14,-4246.57
1972,ORD-001973,4208.38,8416.76,-4208.38
7460,ORD-007461,4116.56,8233.12,-4116.56
903,ORD-000904,4032.33,8064.66,-4032.33
6125,ORD-006126,2097.06,5909.89,-3812.83
1661,ORD-001662,3734.64,7469.28,-3734.64
1390,ORD-001391,3513.77,7027.54,-3513.77
7729,ORD-007730,3488.23,6976.46,-3488.23
8917,ORD-008918,3448.74,6897.48,-3448.74


In [58]:
# ==========================================
# TRACE PAYMENT DOUBLE COUNTING
# ==========================================

payment_trace = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        event_id,
        event_type,
        "payload.amount" AS payment_amount,
        occurred_at_utc
    FROM silver.payment_events
    WHERE "payload.order_id" IN (
        'ORD-001091',
        'ORD-007461',
        'ORD-001973',
        'ORD-000904',
        'ORD-006126'
    )
    ORDER BY "payload.order_id", occurred_at_utc
    """,
    engine
)

display(payment_trace)

,order_id,event_id,event_type,payment_amount,occurred_at_utc
0,ORD-000904,EVT-00005579,PAYMENT_AUTHORIZED,4032.33,2026-07-12T01:44:00Z
1,ORD-000904,EVT-00005580,PAYMENT_CAPTURED,4032.33,2026-07-12T02:39:00Z
2,ORD-000904,EVT-00005580,PAYMENT_CAPTURED,4032.33,2026-07-12T02:39:00Z
3,ORD-001091,EVT-00006695,PAYMENT_AUTHORIZED,4246.57,2026-07-09T19:30:00Z
4,ORD-001091,EVT-00006696,PAYMENT_CAPTURED,4246.57,2026-07-09T20:25:00Z
5,ORD-001091,EVT-00006696,PAYMENT_CAPTURED,4246.57,2026-07-09T20:25:00Z
6,ORD-001973,EVT-00012182,PAYMENT_AUTHORIZED,4208.38,2026-09-03T11:50:00Z
7,ORD-001973,EVT-00012183,PAYMENT_CAPTURED,4208.38,2026-09-03T12:45:00Z
8,ORD-001973,EVT-00012183,PAYMENT_CAPTURED,4208.38,2026-09-03T12:45:00Z
9,ORD-006126,EVT-00038067,PAYMENT_AUTHORIZED,3812.83,2026-07-18T22:31:00Z


In [59]:
# ==========================================
# CORRECTED PAYMENT SUMMARY
# ==========================================

corrected_payment_summary = pd.read_sql(
    """
    WITH deduplicated_events AS (
        SELECT DISTINCT
            event_id,
            "payload.order_id" AS order_id,
            event_type,
            "payload.amount" AS payment_amount
        FROM silver.payment_events
    )

    SELECT
        order_id,
        SUM(payment_amount) AS captured_payment_amount
    FROM deduplicated_events
    WHERE event_type = 'PAYMENT_CAPTURED'
    GROUP BY order_id
    """,
    engine
)

print("Corrected payment orders:", len(corrected_payment_summary))

display(corrected_payment_summary.head(20))

Corrected payment orders: 8785


,order_id,captured_payment_amount
0,ORD-000299,676.40
1,ORD-005343,270.54
2,ORD-002601,1553.97
3,ORD-005158,1352.00
4,ORD-006850,406.11
5,ORD-005256,292.58
6,ORD-003129,768.54
7,ORD-008070,1932.75
8,ORD-009333,341.82
9,ORD-000003,270.99


In [65]:
# ==========================================
# REBUILD PAYMENT FIELDS IN ENRICHED DATA
# ==========================================
import numpy as np
# Hapus payment fields lama
customer_order_enriched = customer_order_enriched.drop(
    columns=[
        "captured_payment_amount",
        "payment_status"
    ],
    errors="ignore"
)

# Merge corrected payment summary
customer_order_enriched = customer_order_enriched.merge(
    corrected_payment_summary,
    on="order_id",
    how="left"
)

# Order tanpa captured payment = 0
customer_order_enriched["captured_payment_amount"] = (
    customer_order_enriched["captured_payment_amount"]
    .fillna(0)
)

# Tentukan payment status
customer_order_enriched["payment_status"] = np.where(
    customer_order_enriched["captured_payment_amount"] > 0,
    "PAYMENT_CAPTURED",
    "PAYMENT_AUTHORIZED"
)

print("Payment enrichment updated.")
print("Rows:", len(customer_order_enriched))
print(
    "Unique orders:",
    customer_order_enriched["order_id"].nunique()
)

display(
    customer_order_enriched[
        [
            "order_id",
            "captured_payment_amount",
            "payment_status"
        ]
    ].head(20)
)

Payment enrichment updated.
Rows: 10000
Unique orders: 10000


,order_id,captured_payment_amount,payment_status
0,ORD-000299,676.40,PAYMENT_CAPTURED
1,ORD-005343,270.54,PAYMENT_CAPTURED
2,ORD-005158,1352.00,PAYMENT_CAPTURED
3,ORD-002601,1553.97,PAYMENT_CAPTURED
4,ORD-006850,406.11,PAYMENT_CAPTURED
5,ORD-005256,292.58,PAYMENT_CAPTURED
6,ORD-003129,768.54,PAYMENT_CAPTURED
7,ORD-008070,1932.75,PAYMENT_CAPTURED
8,ORD-009333,341.82,PAYMENT_CAPTURED
9,ORD-004295,428.77,PAYMENT_CAPTURED


In [66]:
display(
    customer_order_enriched[
        customer_order_enriched["order_id"] == "ORD-003129"
    ][
        [
            "order_id",
            "captured_payment_amount",
            "payment_status",
            "net_revenue"
        ]
    ]
)

,order_id,captured_payment_amount,payment_status,net_revenue
6,ORD-003129,768.54,PAYMENT_CAPTURED,783.54


In [60]:
# ==========================================
# VALIDATE CORRECTED PAYMENT
# ==========================================

corrected_payment_compare = customer_order_enriched[
    [
        "order_id",
        "captured_payment_amount"
    ]
].merge(
    corrected_payment_summary,
    on="order_id",
    how="left",
    suffixes=("_enriched", "_corrected")
)

corrected_payment_compare["captured_payment_amount_corrected"] = (
    corrected_payment_compare[
        "captured_payment_amount_corrected"
    ].fillna(0)
)

corrected_payment_compare["difference"] = (
    corrected_payment_compare["captured_payment_amount_enriched"]
    - corrected_payment_compare["captured_payment_amount_corrected"]
).round(2)

corrected_mismatch = corrected_payment_compare[
    corrected_payment_compare["difference"] != 0
]

print("Total orders:", len(corrected_payment_compare))
print("Corrected payment mismatches:", len(corrected_mismatch))

display(corrected_mismatch.head(20))

Total orders: 10000
Corrected payment mismatches: 1915


,order_id,captured_payment_amount_enriched,captured_payment_amount_corrected,difference
6,ORD-003129,422.70,768.54,-345.84
13,ORD-003087,101.45,184.45,-83.00
17,ORD-007999,3192.87,5805.22,-2612.35
18,ORD-005404,1010.50,1837.28,-826.78
19,ORD-000303,786.72,1430.40,-643.68
22,ORD-008749,70.69,128.52,-57.83
38,ORD-009258,887.05,1612.82,-725.77
49,ORD-007290,1011.99,1839.98,-827.99
50,ORD-003742,774.13,1407.50,-633.37
54,ORD-009017,1808.25,3287.72,-1479.47


In [61]:
print(
    "Enriched total:",
    round(
        customer_order_enriched["captured_payment_amount"].sum(),
        2
    )
)

print(
    "Corrected payment total:",
    round(
        corrected_payment_summary["captured_payment_amount"].sum(),
        2
    )
)

Enriched total: 11815776.97
Corrected payment total: 13137988.14


In [62]:
# ==========================================
# TRACE SINGLE PAYMENT MISMATCH
# ==========================================

payment_trace = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        event_id,
        event_type,
        "payload.amount" AS payment_amount,
        occurred_at_utc
    FROM silver.payment_events
    WHERE "payload.order_id" = 'ORD-003129'
    ORDER BY occurred_at_utc
    """,
    engine
)

display(payment_trace)

,order_id,event_id,event_type,payment_amount,occurred_at_utc
0,ORD-003129,EVT-00019337,PAYMENT_AUTHORIZED,768.54,2026-07-05T17:46:00Z
1,ORD-003129,EVT-00019339,PAYMENT_CAPTURED,345.84,2026-07-05T19:41:00Z
2,ORD-003129,EVT-00019338,PAYMENT_CAPTURED,422.70,2026-07-06T01:41:00+07:00


In [63]:
enriched_payment_trace = customer_order_enriched[
    customer_order_enriched["order_id"] == "ORD-003129"
][
    [
        "order_id",
        "captured_payment_amount",
        "payment_status",
        "net_revenue"
    ]
]

display(enriched_payment_trace)

,order_id,captured_payment_amount,payment_status,net_revenue
6,ORD-003129,422.7,PAYMENT_CAPTURED,783.54
